In [2]:
import sys
import os
import json
import tensorflow as tf
import numpy as np

# 1. Add your project root to Python path so we can import 'gan'
# Change this to the actual path of your 'Zihao' or 'zzGAN' folder
PROJECT_ROOT = '/project/animesh_ray_1465/Zihao/GAN/zzGAN' 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. Import your updated model directly
from gan.sngan.generator_gumbel import GumbelGenerator

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Flags initialized. Sequence Length: 160, Vocab: 21


In [3]:
class FakeFlags:
    # --- Architecture matches your updated files ---
    model_type = 'wgan'
    architecture = 'gumbel'
    batch_size = 64
    z_dim = 128
    gf_dim = 64
    df_dim = 64
    
    # Kernel / Dilation / Attention
    kernel_height = 3
    kernel_width = 3
    dilation_rate = 2
    attn_pos = 2
    
    # Sequence Config
    seq_len = 160
    vocab_size = 21   
    
    # Misc
    dataset = 'zz'
    logdir = '/project/animesh_ray_1465/Zihao/GAN/logs'
    
    # These values don't affect inference, but are needed to init the class
    generator_learning_rate = 1e-4
    discriminator_learning_rate = 5e-5
    beta1 = 0.5
    beta2 = 0.9
    multid_schedule = 20000
    d_step = 3
    fm_weight = 10
    lambda_gp = 10

    def __getattr__(self, name):
        # Fallback for any flag accessed by code that I didn't explicitly define
        return None

FLAGS = FakeFlags()
print(f"Config loaded. Seq Len: {FLAGS.seq_len}")

In [ ]:
def load_and_generate(run_dir, checkpoint_step=None, num_samples=50):
    """
    Loads the generator from the source code + checkpoint.
    """
    # 1. Initialize Architecture from your imported module
    #    The imported class already has RefineBlocks/GPS logic inside it.
    dummy_shape = [1, 1, FLAGS.seq_len, FLAGS.vocab_size]
    g_model = GumbelGenerator(FLAGS, dummy_shape)
    
    # 2. Force Build (Create Variables)
    #    Crucial: This runs the 'call' method once to create the GPS channel weights
    #    and Lazy-Build layers before we try to load weights into them.
    print("Building model graph...")
    dummy_z = tf.random.normal([1, FLAGS.z_dim])
    _ = g_model(dummy_z, training=False)
    
    # 3. Restore Weights
    #    We recreate the exact checkpoint structure used in training:
    #    ckpt = tf.train.Checkpoint(generator=g_model, ...)
    ckpt = tf.train.Checkpoint(generator=g_model)
    
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=5)
    
    if checkpoint_step:
        # e.g., load specific step 30000
        ckpt_path = f"{ckpt_dir}/ckpt-{checkpoint_step}"
    else:
        # Load latest
        ckpt_path = manager.latest_checkpoint

    if ckpt_path:
        status = ckpt.restore(ckpt_path)
        # Expect Partial: We are loading ONLY generator, ignoring D/Optimizer in ckpt
        status.expect_partial() 
        print(f"SUCCESS: Restored weights from {ckpt_path}")
    else:
        print(f"CRITICAL: No checkpoint found in {ckpt_dir}")
        return None

    # 4. Generate
    print(f"Generating {num_samples} sequences...")
    z = tf.random.normal([num_samples, FLAGS.z_dim])
    probs = g_model(z, training=False)
    
    # 5. Decode
    vocab = "ACDEFGHIKLMNPQRSTVWY" # Residues 1-20
    seqs = []
    for i in range(num_samples):
        indices = tf.argmax(probs[i], axis=-1).numpy().flatten()
        # Clean decode: ignore 0 (pad), join others
        valid = [vocab[idx-1] for idx in indices if idx != 0]
        seqs.append("".join(valid))
        
    return seqs

In [ ]:
# UPDATE THIS PATH to your trained folder
# e.g., '/project/animesh_ray_1465/Zihao/GAN/logs/trial4/20231025-XXXXXX'
TARGET_RUN_DIR = '/project/animesh_ray_1465/Zihao/GAN/logs/trial4/YOUR_TIMESTAMP_HERE'

# Optional: Load a specific step (e.g., 30 for step 30k if your naming convention is simple index)
# Or leave None for latest.
seqs = load_and_generate(TARGET_RUN_DIR, checkpoint_step=None, num_samples=20)

if seqs:
    print("\n--- GENERATED SEQUENCES ---")
    for i, s in enumerate(seqs):
        print(f">{i}\n{s}")